### Programación para Ciencia de Datos

### Parte 1: Análisis Estadístico Básico 30

### Parte 2: Indicadores Técnicos 35

### Parte 3: Sistema de Alertas 35

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional

In [ ]:
def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadísticas descriptivas de los precios.
    """
    resultado = {
        "precio_actual": precios.iloc[-1],
        "precio_minimo": precios.min(),
        "precio_maximo": precios.max(),
        "precio_promedio": precios.mean(),
        "precio_mediana": precios.median(),
        "desviacion_std": precios.std(),
        "rango": precios.max() - precios.min(),
        "dias_analizados": len(precios)
    }
    return resultado

In [ ]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    """
    return precios.pct_change() * 100

In [ ]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:
    """
    Analiza los rendimientos calculados.
    """
    rend_limpios = rendimientos.dropna()
    resultado = {
        "rendimiento_total": rend_limpios.sum(),
        "rendimiento_promedio": rend_limpios.mean(),
        "mejor_dia": (rend_limpios.idxmax().strftime('%Y-%m-%d'), rend_limpios.max()),
        "peor_dia": (rend_limpios.idxmin().strftime('%Y-%m-%d'), rend_limpios.min()),
        "dias_positivos": int((rend_limpios > 0).sum()),
        "dias_negativos": int((rend_limpios < 0).sum()),
        "volatilidad": rend_limpios.std()
    }
    return resultado

In [ ]:
def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media móvil simple (SMA).
    """
    return precios.rolling(window=ventana).mean()

In [ ]:
def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """
    Calcula las Bandas de Bollinger.
    """
    banda_media = precios.rolling(window=ventana).mean()
    std_movil = precios.rolling(window=ventana).std()
    resultado = {
        "banda_superior": banda_media + num_std * std_movil,
        "banda_media": banda_media,
        "banda_inferior": banda_media - num_std * std_movil
    }
    return resultado

In [ ]:
def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> Dict:
    """
    Detecta máximos y mínimos locales.
    """
    max_rolling = precios.rolling(window=2 * ventana + 1, center=True).max()
    min_rolling = precios.rolling(window=2 * ventana + 1, center=True).min()

    mascara_max = precios == max_rolling
    mascara_min = precios == min_rolling

    resultado = {
        "maximos": precios[mascara_max],
        "minimos": precios[mascara_min]
    }
    return resultado

In [ ]:
def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.
    """
    ma = media_movil(precios, ventana).dropna()
    if len(ma) < 2:
        return "LATERAL"

    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2]

    ma_subiendo = ma_actual > ma_anterior
    precio_sobre_ma = precio_actual > ma_actual

    if precio_sobre_ma and ma_subiendo:
        return "ALCISTA"
    elif not precio_sobre_ma and not ma_subiendo:
        return "BAJISTA"
    else:
        return "LATERAL"

In [ ]:
def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera señales de compra/venta basadas en cruces de medias móviles.
    """
    mac = media_movil(precios, ma_corta)
    mal = media_movil(precios, ma_larga)

    cruce_alcista = (mac.shift(1) < mal.shift(1)) & (mac >= mal)
    cruce_bajista = (mac.shift(1) > mal.shift(1)) & (mac <= mal)

    senales = pd.Series("MANTENER", index=precios.index)
    senales[cruce_alcista] = "COMPRA"
    senales[cruce_bajista] = "VENTA"
    return senales

In [ ]:
def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> List[Dict]:
    """
    Genera alertas cuando hay cambios significativos.
    """
    rendimientos = calcular_rendimientos(precios).dropna()
    alertas = []
    for fecha, cambio in rendimientos.items():
        if abs(cambio) >= umbral_cambio:
            alertas.append({
                "fecha": fecha.strftime('%Y-%m-%d') if hasattr(fecha, 'strftime') else str(fecha),
                "tipo": "SUBIDA" if cambio > 0 else "CAIDA",
                "cambio": cambio
            })
    return alertas

In [ ]:
def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.
    """
    std = rendimientos.dropna().std()
    if std < 1:
        return "BAJA"
    elif std < 3:
        return "MEDIA"
    elif std < 5:
        return "ALTA"
    else:
        return "MUY ALTA"

In [ ]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de análisis.
    """
    rendimientos = calcular_rendimientos(precios)
    senales = generar_senales_trading(precios)

    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": precios.index[0].strftime('%Y-%m-%d'),
            "fin": precios.index[-1].strftime('%Y-%m-%d'),
            "dias": len(precios)
        },
        "estadisticas": estadisticas_basicas(precios),
        "rendimientos": analisis_rendimientos(rendimientos),
        "tendencia": clasificar_tendencia(precios),
        "volatilidad": clasificar_volatilidad(rendimientos),
        "senal_actual": senales.iloc[-1],
        "alertas_recientes": alertas_precio(precios)
    }
    return reporte

### Datos de Prueba

In [ ]:
np.random.seed(42)  # Para reproducibilidad

fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')  # B = días hábiles

precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)  # Media 0.2%, std 2%
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 días):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de días: {len(PRECIOS_ACCION)}")

In [ ]:
np.random.seed(123)

rend_volatil = np.random.normal(0, 0.05, 60)  # 5% de volatilidad diaria
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

rend_bajista = np.random.normal(-0.005, 0.015, 60)  # Tendencia negativa
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para análisis:")
print(f"1. ACME Corp - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

In [ ]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible."""
    print("=" * 70)
    print(f"           REPORTE DE ANÁLISIS: {reporte['nombre']}")
    print("=" * 70)
    
    periodo = reporte.get('periodo', {})
    print(f"\n📅 PERÍODO DE ANÁLISIS")
    print("-" * 40)
    print(f"Inicio: {periodo.get('inicio', 'N/A')}")
    print(f"Fin: {periodo.get('fin', 'N/A')}")
    print(f"Días analizados: {periodo.get('dias', 'N/A')}")
    
    stats = reporte.get('estadisticas', {})
    print(f"\n📊 ESTADÍSTICAS DE PRECIO")
    print("-" * 40)
    print(f"Precio actual:  ${stats.get('precio_actual', 0):,.2f}")
    print(f"Precio mínimo:  ${stats.get('precio_minimo', 0):,.2f}")
    print(f"Precio máximo:  ${stats.get('precio_maximo', 0):,.2f}")
    print(f"Precio promedio: ${stats.get('precio_promedio', 0):,.2f}")
    
    rend = reporte.get('rendimientos', {})
    print(f"\n📈 RENDIMIENTO")
    print("-" * 40)
    print(f"Rendimiento total: {rend.get('rendimiento_total', 0):+.2f}%")
    print(f"Rendimiento promedio diario: {rend.get('rendimiento_promedio', 0):+.3f}%")
    if rend.get('mejor_dia'):
        print(f"Mejor día: {rend['mejor_dia'][0]} ({rend['mejor_dia'][1]:+.2f}%)")
    if rend.get('peor_dia'):
        print(f"Peor día: {rend['peor_dia'][0]} ({rend['peor_dia'][1]:+.2f}%)")
    print(f"Días positivos: {rend.get('dias_positivos', 0)}")
    print(f"Días negativos: {rend.get('dias_negativos', 0)}")
    
    print(f"\n🎯 INDICADORES")
    print("-" * 40)
    print(f"Tendencia: {reporte.get('tendencia', 'N/A')}")
    print(f"Volatilidad: {reporte.get('volatilidad', 'N/A')}")
    print(f"Señal actual: {reporte.get('senal_actual', 'N/A')}")
    
    alertas = reporte.get('alertas_recientes', [])
    if alertas:
        print(f"\n⚠️ ALERTAS RECIENTES")
        print("-" * 40)
        for alerta in alertas[-5:]:  # Últimas 5
            emoji = "🔺" if alerta['tipo'] == 'SUBIDA' else "🔻"
            print(f"{emoji} {alerta['fecha']}: {alerta['tipo']} de {alerta['cambio']:+.2f}%")
    
    print("\n" + "=" * 70)

In [ ]:
def visualizar_precios_texto(precios: pd.Series, ancho: int = 50) -> None:
    """Visualización simple de precios en texto (ASCII chart)."""
    min_precio = precios.min()
    max_precio = precios.max()
    rango = max_precio - min_precio
    
    print(f"\nGráfico de precios: {precios.name}")
    print(f"Max: ${max_precio:.2f}")
    print("-" * (ancho + 10))
    
    for fecha, precio in precios.iloc[::3].items():
        posicion = int((precio - min_precio) / rango * ancho) if rango > 0 else ancho // 2
        barra = " " * posicion + "█"
        fecha_str = fecha.strftime('%m/%d') if hasattr(fecha, 'strftime') else str(fecha)[:5]
        print(f"{fecha_str} |{barra}")
    
    print("-" * (ancho + 10))
    print(f"Min: ${min_precio:.2f}")

In [ ]:
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

print("\n-- Estadísticas Básicas --")
stats = estadisticas_basicas(PRECIOS_ACCION)
print(stats)

print("\n-- Rendimientos (primeros 5) --")
rendimientos = calcular_rendimientos(PRECIOS_ACCION)
print(rendimientos.head())

print("\n-- Análisis de Rendimientos --")
analisis = analisis_rendimientos(rendimientos)
print(analisis)

In [ ]:
print("\n-- Media Móvil (5 días) --")
ma5 = media_movil(PRECIOS_ACCION, 5)
print(ma5.tail())

print("\n-- Bandas de Bollinger --")
bandas = bandas_bollinger(PRECIOS_ACCION, 20, 2)
for nombre, serie in bandas.items():
    if serie is not None:
        print(f"{nombre}: {serie.iloc[-1]:.2f}")

print("\n-- Tendencia --")
tendencia = clasificar_tendencia(PRECIOS_ACCION)
print(f"Tendencia actual: {tendencia}")

In [ ]:
print("\nGENERANDO REPORTE COMPLETO...\n")
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte)

In [ ]:
print("\n" + "=" * 70)
print("         COMPARACIÓN DE ACCIONES")
print("=" * 70)

acciones = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

for precios, nombre in acciones:
    rendimientos = calcular_rendimientos(precios)
    if rendimientos is not None:
        rend_total = rendimientos.sum() if not rendimientos.isna().all() else 0
        volatilidad = clasificar_volatilidad(rendimientos)
        tendencia = clasificar_tendencia(precios)
        
        print(f"\n{nombre}:")
        print(f"  Rendimiento: {rend_total:+.2f}%")
        print(f"  Volatilidad: {volatilidad}")
        print(f"  Tendencia: {tendencia}")

### Bonus: Funcionalidades Extra Opcional

In [ ]:
def calcular_rsi(precios: pd.Series, periodos: int = 14) -> pd.Series:
    """
    Calcula el RSI (Relative Strength Index).
    RSI = 100 - (100 / (1 + RS))
    RS = Promedio de ganancias / Promedio de pérdidas
    """
    delta = precios.diff()
    ganancias = delta.clip(lower=0)
    perdidas = -delta.clip(upper=0)

    avg_ganancia = ganancias.rolling(window=periodos).mean()
    avg_perdida = perdidas.rolling(window=periodos).mean()

    rs = avg_ganancia / avg_perdida
    rsi = 100 - (100 / (1 + rs))
    return rsi

rsi = calcular_rsi(PRECIOS_ACCION)
print("📊 RSI (últimos 5 días):")
print(rsi.tail())
print(f"\nRSI actual: {rsi.iloc[-1]:.2f}")
if rsi.iloc[-1] > 70:
    print("⚠️  Señal: SOBRECOMPRADO")
elif rsi.iloc[-1] < 30:
    print("⚠️  Señal: SOBREVENDIDO")
else:
    print("✅ RSI en zona neutral")

In [ ]:
def backtest_estrategia(precios: pd.Series, senales: pd.Series, capital_inicial: float = 10000) -> Dict:
    """
    Simula la estrategia de trading y calcula rendimiento.
    """
    capital = capital_inicial
    acciones = 0
    num_operaciones = 0
    operaciones_ganadoras = 0
    precio_compra = None

    for fecha in precios.index:
        precio = precios[fecha]
        senal = senales[fecha]

        if senal == "COMPRA" and acciones == 0 and capital > 0:
            acciones = capital / precio
            capital = 0
            precio_compra = precio
            num_operaciones += 1

        elif senal == "VENTA" and acciones > 0:
            capital = acciones * precio
            if capital > precio_compra * (capital_inicial / precio_compra if precio_compra else 1):
                operaciones_ganadoras += 1
            acciones = 0
            precio_compra = None

    if acciones > 0:
        capital = acciones * precios.iloc[-1]

    rendimiento_total = ((capital - capital_inicial) / capital_inicial) * 100

    return {
        "capital_final": round(capital, 2),
        "rendimiento_total": round(rendimiento_total, 2),
        "num_operaciones": num_operaciones,
        "operaciones_ganadoras": operaciones_ganadoras
    }

senales_acme = generar_senales_trading(PRECIOS_ACCION)
resultado_bt = backtest_estrategia(PRECIOS_ACCION, senales_acme)
print("💰 BACKTESTING - ACME Corp")
print("-" * 40)
print(f"Capital inicial:   $10,000.00")
print(f"Capital final:     ${resultado_bt['capital_final']:,.2f}")
print(f"Rendimiento total: {resultado_bt['rendimiento_total']:+.2f}%")
print(f"Operaciones:       {resultado_bt['num_operaciones']}")
print(f"Ganadoras:         {resultado_bt['operaciones_ganadoras']}")